# 1. STT(Speech to Text)

## 1) OpenAI API

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI

client = OpenAI()

In [3]:
with open("./audios/my_voice.mp3", "rb") as f:
    result = client.audio.transcriptions.create(
        model = "gpt-4o-mini-transcribe",
        file=f,
        language="ko"
    )

print(result)

Transcription(text='안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', logprobs=None, usage=UsageTokens(input_tokens=53, output_tokens=18, total_tokens=71, type='tokens', input_token_details=UsageTokensInputTokenDetails(audio_tokens=52, text_tokens=1)))


In [4]:
# 라이브러리 설치 uv add pyaudio speechrecognition pydub
import speech_recognition as sr

r = sr.Recognizer()

with sr.Microphone() as source:
    print("말해주세요")
    r.adjust_for_ambient_noise(source)  # 주변 소음 설정
    # STEP 1. 마이크 입력 받기
    audio = r.listen(source)
    # STEP 2. 텍스트 변환
    print("인식 중입니다....")
    text = r.recognize_openai(audio)
    print(f"인식된 텍스트: {text}")
    # STEP 3. 오디오 저장
    audio_file = audio.get_wav_data()
    with open("./audios/real_voice.wav", "wb") as f:
        f.write(audio_file)
    print("목소리 저장 완료!")

말해주세요
인식 중입니다....
인식된 텍스트: 안녕히계세요.
목소리 저장 완료!


In [12]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_wav("./audios/real_voice.wav")

play(sound)

## 2) 로컬 Whisper 모델

In [7]:
# whisper github 검색
# 라이브러리 설치 uv add openai-whisper
import whisper

# 모델
model = whisper.load_model("base")

# 텍스트 변환
result = model.transcribe(
    "./audios/my_voice.mp3",
    language = "ko",
    fp16=True   # CPU 사용할 거라면 False
)

print(result)
print(f"인식된 텍스트 : {result["text"]}")

{'text': ' 안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 3.68, 'text': ' 안녕하세요. 개그우먼 이수지입니다.', 'tokens': [50364, 19289, 13, 14552, 22069, 22471, 48150, 4329, 230, 246, 1831, 7416, 13, 50548], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}, {'id': 1, 'seek': 0, 'start': 3.68, 'end': 5.18, 'text': ' 반갑습니다.', 'tokens': [50548, 16396, 27358, 3115, 13, 50623], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}], 'language': 'ko'}
인식된 텍스트 :  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.


In [10]:
# File Not Found
import os
print(os.getcwd())

c:\Users\user\Documents\pj03


## 3) Faster-Whisper 모델

In [13]:
# faster-whisper github 검색
# 라이브러리 설치 uv add faster-whisper
from faster_whisper import WhisperModel

# 모델
model = WhisperModel(
    "base",
    device="cuda",
    compute_type="float16"
)

# 텍스트 변환
segments, info = model.transcribe(
    "./audios/my_voice.mp3",
    language="ko",
    beam_size=5 # 높을수록 정확하지만 느려진다(default = 5)
)

print(segments)
full_text = ""
for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")
    full_text += seg.text
print(f"인식된 텍스트 : {full_text}")
print(f"감지된 언어: {info.language} (확률: {info.language_probability: .0%})")

<generator object WhisperModel.generate_segments at 0x000001EDCB35BFE0>
[0.00s -> 3.68s]  안녕하세요. 개그우먼 이수지입니다.
[3.68s -> 5.18s]  반갑습니다.
인식된 텍스트 :  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.
감지된 언어: ko (확률:  100%)


# 2. TTS(Test to Speech)

## 1) OpenAI API

In [ ]:
# http://www.openai.fm/ 목소리탐험

In [ ]:
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model = "gpt-4o-mini-tts",
    voice="verse",
    input="Actually 이젠 정말 헤어졌으면 좋겠어. 잠깐 말고 정식으로 한 쪽만 손해 본 걸",
    speed=1.2,  # 0.25~ 4.0
    instructions="밝고 자신감 있는 목소리로 말해줘."
) as response:
    response.stream_to_file("./audios/speech.mp3")

In [35]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audios/speech.mp3")
play(sound)

## 2) ElevenLabs API

In [ ]:
# 로그인 http://elevenlabs.io/
# 라이브러리 설치 uv add elevenlabs

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os
from elevenlabs.client import ElevenLabs

client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))

In [7]:
# 음성 목록 가져오기
response = client.voices.search()

for voice in response.voices:
    print(voice)

voice_id='xi3rF0t7dg7uN2M0WUhr' name='Yuna' samples=None category='professional' fine_tuning=FineTuningResponse(is_allowed_to_fine_tune=True, state={'eleven_turbo_v2_5': 'fine_tuned', 'eleven_v2_5_flash': 'fine_tuned', 'eleven_multilingual_v2': 'fine_tuned', 'eleven_multilingual_sts_v2': 'fine_tuned', 'eleven_flash_v2_5': 'fine_tuned'}, verification_failures=[], verification_attempts_count=0, manual_verification_requested=False, language='ko', progress={}, message={'eleven_turbo_v2_5': '', 'eleven_v2_5_flash': '', 'eleven_multilingual_v2': '', 'eleven_multilingual_sts_v2': '', 'eleven_flash_v2_5': ''}, dataset_duration_seconds=None, verification_attempts=None, slice_ids=None, manual_verification=None, max_verification_attempts=0, next_max_verification_attempts_reset_unix_ms=0, finetuning_state=None) labels={'use_case': 'narrative_story', 'gender': 'female', 'accent': 'standard', 'age': 'young', 'language': 'ko', 'descriptive': 'soft'} description='Yuna - Young Korean female voice with 

In [ ]:
# 음성 변환
audio = client.text_to_speech.convert(
    text="Actually 이젠 정말 헤어졌으면 좋겠어. 잠깐 말고 정식으로 한 쪽만 손해 본 걸",
    voice_id="cgSgspJ2msm6clMCkdW9",
    model_id="eleven_multilingual_v2",
    output_format="mp3_44100_128", 
    voice_settings={
        "stability": 0.3,   # 감정의 변화 정도 (낮을 수록 감정 풍부)
        "similarity_boost": 0.8,    # 원 화자와의 유사도 (높을 수록 비슷한 목소리)
        "use_speaker_boost": True   # 화자 억양 향상
    }
)

file_path = "./audios/voice_clone_jessica.mp3"

audio_bytes = b"".join(audio)

with open(file_path, "wb") as f:
    f.write(audio_bytes)

In [17]:
# 오디오 백그라운드 실행
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audios/voice_clone_jessica.mp3")
play(sound)


In [12]:
## 고윤정 목소리 원본
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audios/koyoonjeong.mp3")
play(sound)

In [13]:
## 고윤정 목소리 클론
from pydub import AudioSegment 
from pydub.playback import play 

sound = AudioSegment.from_mp3("./audios/voice_clone_kyj.mp3")
play(sound)

c:\Users\user\Documents\pj03\.venv\Lib\site-packages\pydub\utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
c:\Users\user\Documents\pj03\.venv\Lib\site-packages\pydub\utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
c:\Users\user\Documents\pj03\.venv\Lib\site-packages\pydub\utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
c:\Users\user\Documents\pj03\.venv\Lib\site-packages\pydub\utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


FileNotFoundError: [Errno 2] No such file or directory: './audios/voice_clone_kyj.mp3'

## 3) Qwen3 TTS

In [ ]:
# 프로젝트 새로 만들어서 할 예정
# 충돌남